# Upload the DPO-aligned Adapter to the Hugging Face Hub

Pushes your final DPO LoRA adapter (+ tokenizer with the chat template) to a repo
like `your-username/customer-support-assistant-dpo`, so the Gradio front end
(`app.py`) can load it straight from the Hub.

**You need:** a free Hugging Face account and a **write** access token
(https://huggingface.co/settings/tokens).

Run on a **T4 GPU** runtime.

## 1. Install

In [ ]:
%%capture
!pip install --upgrade unsloth unsloth_zoo huggingface_hub

## 2. Get your DPO adapter onto this runtime

In [ ]:
# If this is a fresh runtime, upload the dpo_adapter.zip you downloaded earlier.
# (Skip this cell if models/dpo_adapter already exists in the current session.)
import os, shutil
if not os.path.isdir("dpo_adapter") and not os.path.isdir("models/dpo_adapter"):
    from google.colab import files
    up = files.upload()                     # pick dpo_adapter.zip
    zip_name = next(iter(up))
    os.makedirs("dpo_adapter", exist_ok=True)
    shutil.unpack_archive(zip_name, "dpo_adapter")

ADAPTER_DIR = "models/dpo_adapter" if os.path.isdir("models/dpo_adapter") else "dpo_adapter"
print("Using adapter dir:", ADAPTER_DIR, "->", os.listdir(ADAPTER_DIR))

## 3. Log in to Hugging Face

In [ ]:
from huggingface_hub import login

# Paste your WRITE token from https://huggingface.co/settings/tokens
HF_TOKEN = "hf_xxxxxxxxxxxxxxxxxxxxxxxxxxxxxx"   # <-- replace
HF_USERNAME = "your-username"                     # <-- replace
REPO_ID = f"{HF_USERNAME}/customer-support-assistant-dpo"

login(token=HF_TOKEN)
print("Will push to:", REPO_ID)

## 4. Load the adapter and push it to the Hub

In [ ]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = ADAPTER_DIR,       # rebuilds base + your DPO LoRA
    max_seq_length = 2048,
    dtype = None,
    load_in_4bit = True,
)

# Push the LoRA adapter + tokenizer (small, ~a few hundred MB).
model.push_to_hub(REPO_ID, token=HF_TOKEN)
tokenizer.push_to_hub(REPO_ID, token=HF_TOKEN)
print("Done. View it at: https://huggingface.co/" + REPO_ID)

## 5. (Optional) Push a MERGED 16-bit model instead

A merged model is a single standalone model (no separate base needed at load
time), which some deployment targets prefer. It is larger to upload (~6 GB for 3B).

In [ ]:
# Optional - only run if you want a merged model repo as well.
# model.push_to_hub_merged(
#     f"{HF_USERNAME}/customer-support-assistant-dpo-merged",
#     tokenizer, save_method="merged_16bit", token=HF_TOKEN,
# )

### Next
Set the front end to your repo and launch it:
```
HF_MODEL_ID = "your-username/customer-support-assistant-dpo"
```
Then run `app.py` (see the Colab launch cell in the README / app.py header).